In [1]:
import pandas as pd 
import numpy as np

In [2]:
df = pd.read_csv(r"D:\Ashish\Texas\Tarrant\Transaction_Log\Raw_Data\us_tx_48439_data_trxlog_marginal-reference_bronze_20260218.csv")
df

,instrument,marginal_ref_instrument_number,marginal_ref_doc_type,marginal_ref_doc_date,book_page,page_save
0,D206206698,D208398671,RELEASE,10/17/2008,NaN,D206206698.json
1,D206206813,D206216110,LEASE,7/17/2006,NaN,D206206813.json
2,D219100087,D224082481,UCC,5/13/2024,NaN,D219100087.json
3,D206207000,D205088500,DEED OF TRUST,4/1/2005,NaN,D206207000.json
4,D206206998,D204322124,DEED OF TRUST,10/14/2004,NaN,D206206998.json
...,...,...,...,...,...,...
74500,NaN,D204106212,RELEASE,4/12/2004,13795_460,13795_460.json
74501,NaN,D206045794,LEASE,2/15/2006,15967_30,15967_30.json
74502,NaN,D204079892,RELEASE,3/16/2004,14146_182,14146_182.json
74503,NaN,D207441401,LEASE,12/13/2007,13097_272,13097_272.json


## Case-1 Found any column side null equivalnat so replace with null 



In [3]:
# Define fake null values
fake_nulls = ['', 'null', 'none', 'nan', "None", 'na', 'n/a', 'n.a.', 'NaN', 'NAN', 'NULL', 'nil', 'Nan', 'Null',"NONE"]

# ✅ Now check and replace in all columns
for col in df.columns:
    mask = df[col].astype(str).str.lower().isin([str(x).lower() for x in fake_nulls])
    problem_rows = df.loc[mask, [col]]

    if not problem_rows.empty:
        print(f"Rows where '{col}' has fake nulls:")
        print(problem_rows)

        df[col] = df[col].replace(fake_nulls, np.nan)
print("\nNaN count per column:")

print(df.isna().sum())
print("\nCleaned DataFrame:")
print(df)


Rows where 'instrument' has fake nulls:
      instrument
60673        NaN
60674        NaN
60675        NaN
60676        NaN
60677        NaN
...          ...
74500        NaN
74501        NaN
74502        NaN
74503        NaN
74504        NaN

[11713 rows x 1 columns]
Rows where 'book_page' has fake nulls:
      book_page
0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
...         ...
74477       NaN
74478       NaN
74480       NaN
74481       NaN
74482       NaN

[62792 rows x 1 columns]

NaN count per column:
instrument                        11713
marginal_ref_instrument_number        0
marginal_ref_doc_type                 0
marginal_ref_doc_date                 0
book_page                         62792
page_save                             0
dtype: int64

Cleaned DataFrame:
       instrument marginal_ref_instrument_number marginal_ref_doc_type  \
0      D206206698                     D208398671               RELEASE   
1      D206206813            

## Case-2 Removing the extra spaces


In [4]:
import re
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else x
    )    

## Checked below column 


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74505 entries, 0 to 74504
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   instrument                      62792 non-null  object
 1   marginal_ref_instrument_number  74505 non-null  object
 2   marginal_ref_doc_type           74505 non-null  object
 3   marginal_ref_doc_date           74505 non-null  object
 4   book_page                       11713 non-null  object
 5   page_save                       74505 non-null  object
dtypes: object(6)
memory usage: 3.4+ MB


In [6]:
df

,instrument,marginal_ref_instrument_number,marginal_ref_doc_type,marginal_ref_doc_date,book_page,page_save
0,D206206698,D208398671,RELEASE,10/17/2008,NaN,D206206698.json
1,D206206813,D206216110,LEASE,7/17/2006,NaN,D206206813.json
2,D219100087,D224082481,UCC,5/13/2024,NaN,D219100087.json
3,D206207000,D205088500,DEED OF TRUST,4/1/2005,NaN,D206207000.json
4,D206206998,D204322124,DEED OF TRUST,10/14/2004,NaN,D206206998.json
...,...,...,...,...,...,...
74500,NaN,D204106212,RELEASE,4/12/2004,13795_460,13795_460.json
74501,NaN,D206045794,LEASE,2/15/2006,15967_30,15967_30.json
74502,NaN,D204079892,RELEASE,3/16/2004,14146_182,14146_182.json
74503,NaN,D207441401,LEASE,12/13/2007,13097_272,13097_272.json


In [7]:
df['marginal_ref_doc_date'] = pd.to_datetime(df['marginal_ref_doc_date'], errors='raise')

In [8]:
df

,instrument,marginal_ref_instrument_number,marginal_ref_doc_type,marginal_ref_doc_date,book_page,page_save
0,D206206698,D208398671,RELEASE,2008-10-17,NaN,D206206698.json
1,D206206813,D206216110,LEASE,2006-07-17,NaN,D206206813.json
2,D219100087,D224082481,UCC,2024-05-13,NaN,D219100087.json
3,D206207000,D205088500,DEED OF TRUST,2005-04-01,NaN,D206207000.json
4,D206206998,D204322124,DEED OF TRUST,2004-10-14,NaN,D206206998.json
...,...,...,...,...,...,...
74500,NaN,D204106212,RELEASE,2004-04-12,13795_460,13795_460.json
74501,NaN,D206045794,LEASE,2006-02-15,15967_30,15967_30.json
74502,NaN,D204079892,RELEASE,2004-03-16,14146_182,14146_182.json
74503,NaN,D207441401,LEASE,2007-12-13,13097_272,13097_272.json


In [9]:
# Work on string view for consistency
df_str = df.astype(str)

problem_columns_by_type = {}

# --- find single-character string values ---
single_char_values = (
    df_str
    .where(df.notna())
    .apply(lambda c: c[c.str.len() == 1])
    .stack()
    .unique()
)

# --- values to check (include 0 and 0.0) ---
values_to_check = set(single_char_values) | {"0", "0.0"}

# --- run checks ---
for val in sorted(values_to_check):
    result = (df_str == val).sum()
    result = result[result > 0]

    if result.empty:
        continue

    # Print per-value output
    print(f"\n=== Single-character / zero values ({val}) ===")
    print(result.to_string())

    # Store problem-wise column list
    problem_columns_by_type[f"Value ({val})"] = result.index.tolist()

# --- final summary ---
print("\n=== Problem-wise column lists ===")
for k, v in problem_columns_by_type.items():
    print(f"{k}: {v}")


=== Problem-wise column lists ===


In [10]:
df.duplicated().sum()


np.int64(0)

In [11]:
# Define fake null values
fake_nulls = ['', 'null', 'none', 'nan', "None", 'na', 'n/a', 'n.a.', 'NaN', 'NAN', 'NULL', 'nil', 'Nan', 'Null',"NONE"]

# ✅ Now check and replace in all columns
for col in df.columns:
    mask = df[col].astype(str).str.lower().isin([str(x).lower() for x in fake_nulls])
    problem_rows = df.loc[mask, [col]]

    if not problem_rows.empty:
        print(f"Rows where '{col}' has fake nulls:")
        print(problem_rows)

        df[col] = df[col].replace(fake_nulls, np.nan)
print("\nNaN count per column:")

print(df.isna().sum())
print("\nCleaned DataFrame:")
print(df)


Rows where 'instrument' has fake nulls:
      instrument
60673        NaN
60674        NaN
60675        NaN
60676        NaN
60677        NaN
...          ...
74500        NaN
74501        NaN
74502        NaN
74503        NaN
74504        NaN

[11713 rows x 1 columns]
Rows where 'book_page' has fake nulls:
      book_page
0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
...         ...
74477       NaN
74478       NaN
74480       NaN
74481       NaN
74482       NaN

[62792 rows x 1 columns]

NaN count per column:
instrument                        11713
marginal_ref_instrument_number        0
marginal_ref_doc_type                 0
marginal_ref_doc_date                 0
book_page                         62792
page_save                             0
dtype: int64

Cleaned DataFrame:
       instrument marginal_ref_instrument_number marginal_ref_doc_type  \
0      D206206698                     D208398671               RELEASE   
1      D206206813            

In [12]:
import re
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].apply(
        lambda x: re.sub(r'\s+', ' ', x).strip() if isinstance(x, str) else x
    )

In [13]:
df

,instrument,marginal_ref_instrument_number,marginal_ref_doc_type,marginal_ref_doc_date,book_page,page_save
0,D206206698,D208398671,RELEASE,2008-10-17,NaN,D206206698.json
1,D206206813,D206216110,LEASE,2006-07-17,NaN,D206206813.json
2,D219100087,D224082481,UCC,2024-05-13,NaN,D219100087.json
3,D206207000,D205088500,DEED OF TRUST,2005-04-01,NaN,D206207000.json
4,D206206998,D204322124,DEED OF TRUST,2004-10-14,NaN,D206206998.json
...,...,...,...,...,...,...
74500,NaN,D204106212,RELEASE,2004-04-12,13795_460,13795_460.json
74501,NaN,D206045794,LEASE,2006-02-15,15967_30,15967_30.json
74502,NaN,D204079892,RELEASE,2004-03-16,14146_182,14146_182.json
74503,NaN,D207441401,LEASE,2007-12-13,13097_272,13097_272.json


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74505 entries, 0 to 74504
Data columns (total 6 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   instrument                      62792 non-null  object        
 1   marginal_ref_instrument_number  74505 non-null  object        
 2   marginal_ref_doc_type           74505 non-null  object        
 3   marginal_ref_doc_date           74505 non-null  datetime64[ns]
 4   book_page                       11713 non-null  object        
 5   page_save                       74505 non-null  object        
dtypes: datetime64[ns](1), object(5)
memory usage: 3.4+ MB


## Placing the file


In [15]:
df.to_csv(r"D:\Ashish\Texas\Tarrant\Transaction_Log\Cleaned_Data\us_tx_48439_data_trxlog_marginal-reference_silver_20260218.csv",index=False)
